# EGMS kontra GNSS — Groningen (Holandia)

**Pytanie:** czy EGMS (ruch terenu z radaru satelitarnego Sentinel-1) pokazuje takie samo tempo osiadania jak stacje
GNSS na ziemi?

**Dane:**
- EGMS L2b Calibrated, wydanie 2020–2024: ruch punktów w kierunku satelity, osobno dla każdej ścieżki,
- stacje GNSS z Nevada Geodetic Laboratory: dobowe pozycje anten, nad polem gazowym Groningen.

**Jak porównujemy:** dla każdej stacji bierzemy punkty EGMS w promieniu 100 m, ruch GNSS przeliczamy na kierunek patrzenia
satelity i porównujemy tempo ruchu [mm/rok]. Progi oceny ustaliliśmy przed obliczeniami.

**Jak uruchomić:** menu *Środowisko wykonawcze → Uruchom wszystko* (kilka sekund). Notatnik tylko czyta wyniki z Dysku,
policzone wcześniej w `EGMS_Monitoring.ipynb`. Kod jest ukryty — kliknij tytuł komórki, żeby go zobaczyć.

In [ ]:
#@title Przygotowanie: połączenie z Dyskiem i wczytanie wyników
from google.colab import drive
drive.mount('/content/drive')
import json, subprocess, sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Image, Markdown, HTML
PROJECT_DIR = '/content/drive/MyDrive/GeoWorldLook/egmscheck'
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
for m in [m for m in sys.modules if m.startswith('step_0')]:
    del sys.modules[m]          # zawsze kod z Dysku: moduł zapamiętany w jądrze sprzed aktualizacji dawał KeyError
from step_05_colab_run import (CHECK_CONFIG, region_paths, levelling_check_paths, _levelling_check_examples,
                               _example_sites, _read_sites, _sources)
rt = {'PROJECT_DIR': PROJECT_DIR}
print('Wyniki policzone kodem:', subprocess.run(['git', '-C', PROJECT_DIR, 'log', '-1', '--format=%h z %ci'],
                                               capture_output=True, text=True).stdout.strip())

def pokaz(path, width=850):
    # Brak pliku = błąd (wyniki liczy EGMS_Monitoring.ipynb), nie pusta komórka.
    if not Path(path).exists():
        raise FileNotFoundError(f'Brak {path}: uruchom najpierw notebooks/EGMS_Monitoring.ipynb')
    display(Image(str(path), width=width))

def tabela(df):
    display(HTML(df.to_html(index=False)))

def ocena(x):
    s = str(x)
    return '✔ tak' if s == 'True' else '✘ nie' if s == 'False' else '— nie liczone'

def kryteria(path, kody):
    t = pd.read_csv(path)
    t = t[t['kryterium'].str[:2].isin(kody)]
    return pd.DataFrame({'kod': t['kryterium'].str[:2].to_numpy(), 'Kryterium': t['kryterium'].to_numpy(),
                         'Wynik': t['wynik'].to_numpy(), 'Próg': t['próg'].to_numpy(),
                         'Spełnione': t['spełnione'].map(ocena).to_numpy()})
G = region_paths(rt, CHECK_CONFIG, 'groningen')

## 1. Ile mamy danych

In [ ]:
#@title Liczba stacji i par stacja–ścieżka
sites = _read_sites(G)
gn = sites[sites['site_type'] == 'gnss']
gc = pd.read_csv(G['gnss_compare'], dtype={'site': str, 'track': str})
par = gc[(gc['product'] == _sources(CHECK_CONFIG)['L2B']) & (gc['status'] == 'ok')
         & (gc['radius_m'] == CHECK_CONFIG['method']['radius_main_m'])]
display(Markdown(f"- **{len(gn)} stacji GNSS**: {int((gn['group'] == 'osiadająca').sum())} osiadających "
                 f"i {int((gn['group'] == 'stabilna').sum())} stabilnych\n"
                 f"- **{len(par)} par stacja–ścieżka** z kompletem danych (promień {CHECK_CONFIG['method']['radius_main_m']} m)"))

## 2. Czy tempo osiadania się zgadza?

In [ ]:
#@title Wykres: tempo ruchu z EGMS kontra z GNSS
pokaz(G['figures'] / 'groningen_velocity_scatter.png', 620)

**Jak czytać:** każdy punkt to jedna stacja widziana z jednej ścieżki satelity. Oś pozioma to GNSS, pionowa EGMS,
obie w mm/rok. Ukośna linia oznacza idealną zgodność, a szary pas ±1,5 mm/rok to dopuszczalna różnica. Punkty pełne to
stacje osiadające, puste to stacje stabilne.

Prawie wszystkie punkty leżą w szarym pasie. Punkty ze ścieżek wznoszących (niebieskie) są nieco nad linią, a ze
zstępujących (pomarańczowe) nieco pod nią. To mała różnica ruchu poziomego wschód–zachód między GNSS a EGMS, w granicach
progu; nie badaliśmy jej osobno.

## 3. Ocena według kryteriów ustalonych przed obliczeniami

In [ ]:
#@title Tabela kryteriów K1–K3
t = kryteria(G['verdict_l2b'], ['K1', 'K2', 'K3'])
pytania = {'K1': 'Czy tempo osiadania się zgadza?', 'K2': 'Czy szeregi w czasie się zgadzają?',
           'K3': 'Czy na stacjach stabilnych EGMS nie pokazuje ruchu, którego nie ma?'}
t['Kryterium'] = t['kod'] + ': ' + t['kod'].map(pytania)
tabela(t[['Kryterium', 'Wynik', 'Próg', 'Spełnione']])
display(Markdown('**Wniosek według reguły ustalonej z góry:** ' + G['verdict_l2b'].with_suffix('.txt').read_text().strip()))

**Jak czytać:** K1 i K2 są spełnione, czyli tempo osiadania i przebieg w czasie się zgadzają. K3 nie przeszło o włos
na jednej stacji stabilnej, dlatego wniosek formalny to zgodność częściowa.

## 4. Przykładowa stacja w czasie

In [ ]:
#@title Stacja osiadająca najszybciej według GNSS (wybrana regułą, nie ręcznie)
stacja = _example_sites(sites, None, CHECK_CONFIG['monitoring']['example_rule'])[0]
display(Markdown(f'Stacja **{stacja}**'))
pokaz(G['figures'] / f'groningen_series_{stacja}.png', 900)

**Jak czytać:** każdy panel to jedna ścieżka satelity (ruch w kierunku satelity, LOS) albo ruch pionowy (U).
Linia to EGMS, kropki to dobowe pozycje GNSS przeliczone na ten sam kierunek. Oba szeregi opadają w tym samym tempie.

## Wniosek

- EGMS dobrze oddaje tempo osiadania nad polem gazowym Groningen. To samo pokazują stacje GNSS na ziemi.
- Wniosek formalny to **zgodność częściowa**, bo kontrola na stacjach stabilnych (K3) nie przeszła o włos.
- **Ograniczenie:** EGMS jest kalibrowany modelem z sieci GNSS, więc to porównanie nie jest w pełni niezależne. Dlatego
  drugi notatnik sprawdza EGMS niwelacją, która nie korzysta z GNSS.

**Źródła:** EGMS (Copernicus Land Monitoring Service); GNSS: Nevada Geodetic Laboratory (Blewitt i in. 2016, 2018);
metoda i progi: Koudogbo i in. (2025) EGMS Validation Report, Fabris i Nazari (2026).